# Context Orchestrator Smoke Notebook

Run the orchestrator with either fixture data (no DB/LLM) or real staging DB + LLM. Select the `server/.venv` kernel.


In [ ]:
# Path setup (repo root assumed one level up from server/)
import sys, pathlib
try:
    ROOT = pathlib.Path(__file__).resolve().parents[2]
except NameError:
    cwd = pathlib.Path.cwd().resolve()
    ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
SERVER = ROOT / 'server'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SERVER) not in sys.path:
    sys.path.insert(0, str(SERVER))
print('sys.path head =', sys.path[:3])


In [ ]:
# Toggles
USE_FIXTURES = False   # False => use real runtimes
USE_REAL_DB = True   # True => needs STAGING_DSN or DATABASE_URL
USE_REAL_LLM = True  # True => call OpenAI with plan.messages


In [ ]:
# Imports, env, helpers
import random
import os, asyncio
import json
import asyncpg
import inspect
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI
from types import SimpleNamespace
from uuid import UUID, uuid4

try:
    ENV_PATH = pathlib.Path(__file__).resolve().parents[1] / '.env'
except NameError:
    ENV_PATH = pathlib.Path.cwd().resolve().parents[0] / '.env'
load_dotenv(ENV_PATH, override=False)

async def get_staging_conn():
    dsn = os.getenv('STAGING_DSN') or os.getenv('DATABASE_URL')
    if not dsn:
        raise RuntimeError('Set STAGING_DSN or DATABASE_URL')
    return await asyncpg.connect(dsn)

def call_llm(messages, model='gpt-4o-mini', max_tokens=200, temperature=0.7):
    api_key = os.getenv('OPENAI_API_KEY')
    if not api_key:
        raise RuntimeError('OPENAI_API_KEY required')
    client = OpenAI(api_key=api_key)
    resp = client.chat.completions.create(model=model, messages=messages, max_tokens=max_tokens, temperature=temperature)
    return resp.choices[0].message.content

# Fixture data
FIX_CORE = ('# CORE MEMORIES\n- loves jazz\n- based in NYC', 'builder-mock')
FIX_HISTORY = [
    {'role': 'user', 'content': 'Hi, who are you?'},
    {'role': 'assistant', 'content': 'I am your companion.'},
    {'role': 'user', 'content': 'Can you help with billing?'}
]
FIX_MEMORY = '# MEMORY\n- billing preference: monthly invoice'
FIX_KB = '# KNOWLEDGE\n- Billing docs: https://example.com/billing'
FIX_TOOLS_TRACE = '# TOOL TRACE\n- joke_tool.py: Generate a short joke; accepts optional random_seed integer.'
FIX_TOOLS_RESULT = '# TOOL RESULT\nWhy did the LLM go to therapy? Too many unresolved prompts.'

print('Env loaded from', ENV_PATH)


NOTEBOOK_COMPANION_ID = None
NOTEBOOK_CONVERSATION_ID = None
NOTEBOOK_PROJECT_ID = None
NOTEBOOK_PROJECT_API_KEY = None
API_BASE = "http://localhost:8100"
CORE_MEMORY_TEXT = 'Billing preference: monthly invoice; prefers email summaries.'

async def maybe_add_core_memory_via_api(companion_id):
    global NOTEBOOK_PROJECT_API_KEY
    if not NOTEBOOK_PROJECT_API_KEY:
        print('PROJECT_API_KEY not set; skipping API core memory add (fallback to direct insert).')
        return False
    try:
        import httpx
        url = f"{API_BASE}/v1/companions/{companion_id}/core-memories"
        resp = httpx.post(
            url,
            json={"memories": [CORE_MEMORY_TEXT], "max_total": 50},
            headers={"Authorization": f"Bearer {NOTEBOOK_PROJECT_API_KEY}"},
            timeout=10.0,
        )
        if resp.status_code == 200:
            print('Added core memory via API')
            return True
        else:
            print(f'API core memory add failed: {resp.status_code} {resp.text}')
    except Exception as exc:
        print(f'API core memory add error: {exc}')
    return False

async def ensure_seed_companion(conn):
    """Seed user + project + api key + companion + version + core memory + conversation.
    Returns (companion_id, conversation_id).
    """
    from app.services.api_keys import generate_project_api_key

    global NOTEBOOK_COMPANION_ID, NOTEBOOK_CONVERSATION_ID, NOTEBOOK_PROJECT_ID, NOTEBOOK_PROJECT_API_KEY
    if NOTEBOOK_COMPANION_ID and NOTEBOOK_CONVERSATION_ID:
        return NOTEBOOK_COMPANION_ID, NOTEBOOK_CONVERSATION_ID

    comp_id = uuid4()
    owner_id = uuid4()
    convo_id = uuid4()
    project_id = uuid4()
    email = f"notebook+{random.randint(10000,99999)}@example.com"

    # Seed minimal user
    await conn.execute(
        "INSERT INTO users (id, email, display_name, auth_provider) VALUES ($1, $2, $3, $4)",
        owner_id,
        email,
        'Notebook User',
        'notebook',
    )

    # Seed project and API key
    slug = f"nb-{str(project_id)[:8]}"
    await conn.execute(
        "INSERT INTO projects (id, owner_id, name, slug, is_default, metadata) VALUES ($1, $2, $3, $4, TRUE, '{}'::jsonb)",
        project_id,
        owner_id,
        'Notebook Project',
        slug,
    )
    full_key, prefix, salt, secret_hash = generate_project_api_key()
    await conn.execute(
        "INSERT INTO project_api_keys (id, project_id, created_by, name, prefix, secret_hash, salt, scopes, metadata) VALUES ($1, $2, $3, $4, $5, $6, $7, $8, '{}'::jsonb)",
        uuid4(),
        project_id,
        owner_id,
        'nb-autogen',
        prefix,
        secret_hash,
        salt,
        ['read', 'write'],
    )
    NOTEBOOK_PROJECT_API_KEY = full_key

    cfg = {
        'system_prompt': {
            'full_system_prompt': 'You are a helpful test companion seeded by the notebook.'
        },
        'memory': {'enabled': True},
        'voice': {},
        'context_mode': 'layered',
        'layers': [],
        'context': {
            'max_prompt_tokens': None,
            'target_prompt_fraction': 0.4,
            'reserved_completion_tokens': None,
        },
    }

    await conn.execute(
        "INSERT INTO companions (id, owner_id, name, description, metadata) VALUES ($1, $2, $3, $4, '{}'::jsonb)",
        comp_id,
        owner_id,
        'Notebook Smoke Companion',
        'Seeded automatically by context_orchestrator_smoke.ipynb',
    )

    # Link companion to project
    await conn.execute(
        "INSERT INTO project_companions (project_id, companion_id) VALUES ($1, $2) ON CONFLICT DO NOTHING",
        project_id,
        comp_id,
    )

    await conn.execute(
        "INSERT INTO companion_versions (id, companion_id, version_number, system_prompt, memory_enabled, status, created_at)"
        " VALUES ($1, $2, 1, $3, TRUE, 'DEPLOYED', now())",
        uuid4(),
        comp_id,
        json.dumps(cfg),
    )

    # Add one core memory via API when possible, otherwise direct insert
    api_ok = await maybe_add_core_memory_via_api(comp_id)
    if not api_ok:
        await conn.execute(
            "INSERT INTO memories (id, companion_id, content, is_core, created_at) VALUES ($1, $2, $3, TRUE, now())",
            uuid4(),
            comp_id,
            CORE_MEMORY_TEXT,
        )

    # Create conversation linked to the companion
    ext_user = f"nb-user-{random.randint(1000,9999)}"
    await conn.execute(
        "INSERT INTO conversations (id, external_user_id, companion_id) VALUES ($1, $2, $3)",
        convo_id,
        ext_user,
        comp_id,
    )

    # Seed a small history
    seed_msgs = [
        ('user', "Hi, who are you?"),
        ('assistant', "I'm your notebook test companion."),
        ('user', "Remind me of my billing preference."),
        ('assistant', "You prefer monthly invoices and email summaries."),
    ]
    for role, content in seed_msgs:
        await conn.execute(
            "INSERT INTO messages (id, conversation_id, role, content, input_modality) VALUES ($1, $2, $3, $4, $5)",
            uuid4(),
            convo_id,
            role,
            content,
            'text',
        )

    NOTEBOOK_COMPANION_ID = comp_id
    NOTEBOOK_CONVERSATION_ID = convo_id
    NOTEBOOK_PROJECT_ID = project_id
    print(f'Seeded staging user {owner_id}, project {project_id}, companion {comp_id}, conversation {convo_id}')
    return comp_id, convo_id


In [ ]:
# Import orchestrator and optionally patch with fixtures
try:
    from app.context import orchestrator as orch
    from app.context.schemas import ContextPlan
except ModuleNotFoundError:
    from server.app.context import orchestrator as orch
    from server.app.context.schemas import ContextPlan

class MockConn:
    async def fetch(self, *args, **kwargs): return []
    async def fetchrow(self, *args, **kwargs): return None
    async def execute(self, *args, **kwargs): return 'OK'

class StubRun:
    def __init__(self, key, content):
        self.key = key
        self.content = content
    async def run(self):
        return SimpleNamespace(messages=[{'role': 'system', 'content': self.content}], events=[])

# Stub core/history when not using real DB
if not USE_REAL_DB:
    orch.load_core_prompt = lambda conn, companion_id: asyncio.sleep(0, result=FIX_CORE)
    orch.get_full_history = lambda conn, conversation_id, use_cache=True: asyncio.sleep(0, result=FIX_HISTORY)

if USE_FIXTURES and not USE_REAL_DB:
    orch.MemoryRuntime = lambda **kw: StubRun('memory', FIX_MEMORY)
    orch.KnowledgeRuntime = lambda **kw: StubRun('knowledge', FIX_KB)
    orch.ToolsRuntime = lambda **kw: StubRun('tools', FIX_TOOLS_TRACE + '\n' + FIX_TOOLS_RESULT)
    print('Using fixture patches (no DB/LLM).')
elif not USE_REAL_DB:
    print('Using stub core/history; real runtimes may still call services if enabled.')
else:
    print('Using real orchestrator/runtimes; DB/LLM depend on toggles.')


In [ ]:
# Single-turn runner
async def run_plan(include_memory=False, include_kb=False, include_tools=False, raw=False, user_message='Book me a meeting tomorrow and find related docs', use_real_db=False, conversation_id=None):
    if use_real_db:
        env_comp = os.getenv('STAGING_COMPANION_ID') or os.getenv('COMPANION_ID') or os.getenv('KB_TEST_COMPANION_ID') or os.getenv('TEST_COMPANION_ID')
        env_convo = os.getenv('STAGING_CONVERSATION_ID') or os.getenv('CONVERSATION_ID')
        conn = await get_staging_conn()
        if env_comp:
            companion_id = UUID(str(env_comp))
        else:
            companion_id, _ = await ensure_seed_companion(conn)
        if env_convo:
            conversation_id = UUID(str(env_convo))
        else:
            if conversation_id is None:
                if env_comp:
                    # create a fresh conversation for the provided companion
                    conversation_id = uuid4()
                    await conn.execute(
                        "INSERT INTO conversations (id, external_user_id, companion_id) VALUES ($1, $2, $3)",
                        conversation_id,
                        'nb-user-runtime',
                        companion_id,
                    )
                else:
                    companion_id, conversation_id = await ensure_seed_companion(conn)
        conversation_id = conversation_id or None
    else:
        conn = MockConn()
        companion_id = uuid4()
        conversation_id = conversation_id or uuid4()

    cfg = SimpleNamespace(
        context_mode='raw' if raw else 'layered',
        memory=SimpleNamespace(enabled=True),
        context=SimpleNamespace(tool_summary='email, calendar, search'),
        layers=[
            SimpleNamespace(key='memory', category='memory', enabled=include_memory, params={}),
            SimpleNamespace(key='knowledge_base', category='knowledge_base', enabled=include_kb, params={'gate_strategy': 'keyword'}),
            SimpleNamespace(key='tools', category='tools', enabled=include_tools, params={'gate_strategy': 'keyword'}),
        ],
    )

    append_core = use_real_db
    append_hist = use_real_db
    run_memory = include_memory if use_real_db else False
    run_kb = include_kb if use_real_db else False
    run_tools = include_tools if use_real_db else False

    try:
        extra_kwargs = {}
        try:
            sig = inspect.signature(orch.build_context_plan)
            if 'include_tools' in sig.parameters:
                extra_kwargs['include_tools'] = run_tools
        except Exception:
            pass
        plan: ContextPlan = await orch.build_context_plan(
            conn=conn,
            companion_id=companion_id,
            companion_config=cfg,
            conversation_id=conversation_id,
            user_message=user_message,
            external_user_id='demo-user',
            include_memory=run_memory,
            include_knowledge=run_kb,
            append_core_prompt=append_core,
            append_history=append_hist,
            **extra_kwargs,
        )
    finally:
        if use_real_db:
            await conn.close()

    print('Messages (head):')
    for m in plan.messages:
        head = m.get('content','')[:200]
        print('- {}: {}'.format(m['role'], head))
    print('Events:', [e.name for e in plan.events])
    return plan

sample_prompts = [
    "What's your name?",
    'Summarize our last messages',
    'Find docs about billing limits',
    'Tell me a joke',
]

for prompt in sample_prompts:
    print('=== Prompt:', prompt)
    await run_plan(raw=True, user_message=prompt, use_real_db=USE_REAL_DB)
    await run_plan(include_memory=True, user_message=prompt, use_real_db=USE_REAL_DB)
    await run_plan(include_memory=True, include_kb=True, user_message=prompt, use_real_db=USE_REAL_DB)
    await run_plan(include_memory=True, include_kb=True, include_tools=True, user_message=prompt, use_real_db=USE_REAL_DB)


In [ ]:
# Optional: run LLM on a plan (requires OPENAI_API_KEY)
async def run_and_call_llm(prompt='Tell me a short joke', include_memory=False, include_kb=False, include_tools=False, use_real_db=USE_REAL_DB, conversation_id=None):
    plan = await run_plan(include_memory=include_memory, include_kb=include_kb, include_tools=include_tools, raw=False, user_message=prompt, use_real_db=use_real_db, conversation_id=conversation_id)
    user_msg = {'role': 'user', 'content': prompt}
    llm_messages = plan.messages + [user_msg]
    if USE_REAL_LLM:
        print('--- LLM input messages (last 5) ---')
        for m in llm_messages[-5:]:
            head = m.get('content','')[:200].replace('\n', ' ')
            print(f"- {m.get('role')}: {head}...")
        print('--- LLM output ---')
        print(call_llm(llm_messages))
    else:
        print('LLM call skipped; set USE_REAL_LLM=True to enable.')
    return plan

# Example (uses seeded conversation when real DB is on):
await run_and_call_llm(prompt='Summarize our last messages', include_memory=True, use_real_db=USE_REAL_DB)


## Seed a test companion + core memories, then run memory runtime (real DB)


In [ ]:
import asyncio

async def seed_and_run_memory():
    conn = await get_staging_conn()
    comp_id, convo_id = await ensure_seed_companion(conn)
    await conn.close()
    print(f'Seeded companion {comp_id} with conversation {convo_id}')
    await run_plan(
        include_memory=True,
        user_message='Remind me of my billing preferences',
        use_real_db=True,
        conversation_id=convo_id,
    )

await seed_and_run_memory()


## Ingest a KB markdown and run memory+KB (real services)


In [ ]:
import httpx

API_BASE = API_BASE if 'API_BASE' in globals() else "http://localhost:8100"
PROJECT_API_KEY = NOTEBOOK_PROJECT_API_KEY
COMPANION_ID = str(NOTEBOOK_COMPANION_ID) if 'NOTEBOOK_COMPANION_ID' in globals() and NOTEBOOK_COMPANION_ID else None
MARKDOWN_TEXT = '# Billing Limits

Monthly cap is $500. Overage alerts sent via email.'

async def ingest_markdown():
    if not COMPANION_ID:
        raise RuntimeError('Companion not seeded')
    if not PROJECT_API_KEY:
        raise RuntimeError('PROJECT_API_KEY missing')
    url = f"{API_BASE}/v1/companions/{COMPANION_ID}/knowledge"
    headers = {'Authorization': f'Bearer {PROJECT_API_KEY}'}
    payload = {
        'type': 'markdown',
        'content': MARKDOWN_TEXT,
        'key': None,
        'asset_id': None,
    }
    async with httpx.AsyncClient() as client:
        r = await client.post(url, json=payload, headers=headers, timeout=30)
        r.raise_for_status()
        print('Ingestion response:', r.json())

# Run ingestion (uses seeded companion/project/key)
await ingest_markdown()

# Example live KB+memory run (after ingestion completes)
await run_plan(include_memory=True, include_kb=True, user_message='What are our billing limits?', use_real_db=True)

await run_and_call_llm(
    prompt='What are our billing limits?',
    include_memory=True,
    include_kb=True,
    use_real_db=True,
    conversation_id=NOTEBOOK_CONVERSATION_ID,
)
